# 🛡️ CyberGuard v2: ML-Based Malicious URL Scanner

Walkthrough notebook for the `cyberguard` package. The reusable code lives in `cyberguard/`; this notebook shows the pipeline step by step.

**New in v2:** 36 lexical features (brand impersonation, TLD risk, punycode, entropy...), domain-grouped train/test split (no leakage), 4 models, calibrated probabilities, PR curves, tunable decision threshold, per-URL red-flag explanations, CLI, Streamlit app and tests.

> ⚠️ **Educational project only.** Not a production security tool. Use Google Safe Browsing or similar for real decisions.

## 1. Setup
Run the next cell first: it writes the `cyberguard/` package files, so this notebook works on its own in Google Colab.

In [ ]:
# Writes the `cyberguard/` package next to this notebook, so no zip upload is needed.
import os
FILES = {'cyberguard/__init__.py': '"""CyberGuard: ML-based malicious URL scanner (educational project)."""\n__version__ = "2.0.0"\n\nfrom .features import FEATURE_NAMES, extract_features, features_frame  # noqa: F401\n', 'cyberguard/features.py': '"""Lexical feature extraction. Looks only at URL text, never at the website."""\nfrom __future__ import annotations\n\nimport math\nimport re\nfrom collections import Counter\nfrom urllib.parse import parse_qsl, unquote, urlparse\n\nimport pandas as pd\n\nSUSPICIOUS_WORDS = [\n    "login", "signin", "verify", "update", "secure", "account", "banking", "confirm",\n    "password", "wallet", "bonus", "free", "support", "billing", "unlock", "suspend",\n    "invoice", "recover",\n]\nBRANDS = [\n    "paypal", "google", "microsoft", "apple", "amazon", "netflix", "facebook",\n    "instagram", "whatsapp", "dhl", "fedex", "irs", "sbi", "hdfc", "icici", "linkedin",\n]\nSHORTENERS = {\n    "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd", "buff.ly",\n    "rebrand.ly", "cutt.ly", "shorturl.at",\n}\nSUSPICIOUS_TLDS = {\n    "xyz", "top", "click", "link", "info", "biz", "cc", "ru", "tk", "ml", "ga",\n    "cf", "gq", "work", "zip", "mov", "support", "loan",\n}\n# Small offline list so we need neither the tldextract package nor internet access.\nMULTI_PART_SUFFIXES = {"co.uk", "org.uk", "ac.uk", "gov.uk", "com.au", "co.in", "co.jp", "com.br", "co.nz", "com.cn"}\nIP_RE = re.compile(r"^(?:\\d{1,3}\\.){3}\\d{1,3}$")\nHEX_IP_RE = re.compile(r"^0x[0-9a-f]+$", re.I)\nVOWELS = set("aeiou")\nRISKY_EXT = {".php", ".exe", ".zip", ".scr", ".apk", ".js", ".html", ".htm", ".jar"}\n\nFEATURE_NAMES = [\n    "url_length", "host_length", "path_length", "query_length", "fragment_length",\n    "dot_count", "subdomain_count", "hyphen_in_host", "digit_count", "digit_ratio",\n    "host_digit_count", "letter_ratio", "uppercase_count", "special_char_count",\n    "has_at_symbol", "has_ip_host", "has_port", "suspicious_word_count",\n    "brand_in_subdomain", "brand_in_path", "is_shortener", "path_depth", "query_params",\n    "host_entropy", "path_entropy", "double_slash_in_path", "percent_encoded_count",\n    "is_punycode", "tld_length", "suspicious_tld", "longest_host_token",\n    "host_vowel_ratio", "max_repeated_char_run", "file_extension_risky",\n]\n\n\ndef scheme_free(url: str) -> str:\n    """Drop the http(s):// prefix and a leading www. so every URL is treated alike.\n\n    In popular public datasets the scheme and \'www.\' often differ between the benign and\n    malicious sources (an artifact of how the data was collected), so models learn that\n    shortcut instead of real URL structure. Removing them avoids it.\n    """\n    u = str(url).strip()\n    if "://" in u:\n        u = u.split("://", 1)[1]\n    return u[4:] if u.lower().startswith("www.") else u\n\n\ndef normalize(url: str) -> str:\n    return "http://" + scheme_free(url)\n\n\ndef shannon_entropy(text: str) -> float:\n    if not text:\n        return 0.0\n    n = len(text)\n    return -sum((c / n) * math.log2(c / n) for c in Counter(text).values())\n\n\ndef split_host(host: str) -> tuple[str, str, str]:\n    """Return (subdomain, registered_domain_label, suffix)."""\n    parts = host.split(".") if host else []\n    if len(parts) >= 3 and ".".join(parts[-2:]) in MULTI_PART_SUFFIXES:\n        return ".".join(parts[:-3]), parts[-3], ".".join(parts[-2:])\n    if len(parts) >= 2:\n        return ".".join(parts[:-2]), parts[-2], parts[-1]\n    return "", host, ""\n\n\ndef _longest_run(text: str) -> int:\n    best = run = 0\n    prev = ""\n    for ch in text:\n        run = run + 1 if ch == prev else 1\n        best, prev = max(best, run), ch\n    return best\n\n\ndef extract_features(raw_url: str) -> dict:\n    """Return a dict of numeric features for one URL (all zeros if unparseable)."""\n    raw = str(raw_url).strip()\n    try:\n        url = normalize(raw)\n        parsed = urlparse(url)\n        host = (parsed.hostname or "").lower()\n        path, query, frag = parsed.path or "", parsed.query or "", parsed.fragment or ""\n        lower = unquote(url).lower()\n        try:\n            has_port = int(parsed.port is not None)\n        except ValueError:\n            has_port = 1\n        sub, domain, suffix = split_host(host)\n        tld = suffix.split(".")[-1] if suffix else ""\n        digits = sum(ch.isdigit() for ch in url)\n        letters = sum(ch.isalpha() for ch in url)\n        host_letters = [c for c in host if c.isalpha()]\n        tokens = re.split(r"[.\\-]", host) if host else []\n        last_seg = path.rsplit("/", 1)[-1]\n        ext = "." + last_seg.rsplit(".", 1)[-1].lower() if "." in last_seg else ""\n        return {\n            "url_length": len(url), "host_length": len(host), "path_length": len(path),\n            "query_length": len(query), "fragment_length": len(frag),\n            "dot_count": url.count("."),\n            "subdomain_count": len(sub.split(".")) if sub else 0,\n            "hyphen_in_host": host.count("-"), "digit_count": digits,\n            "digit_ratio": digits / max(len(url), 1),\n            "host_digit_count": sum(c.isdigit() for c in host),\n            "letter_ratio": letters / max(len(url), 1),\n            "uppercase_count": sum(c.isupper() for c in raw),\n            "special_char_count": sum(url.count(c) for c in "@?&=%_~#$"),\n            "has_at_symbol": int("@" in url),\n            "has_ip_host": int(bool(IP_RE.match(host) or HEX_IP_RE.match(host))),\n            "has_port": has_port,\n            "suspicious_word_count": sum(w in lower for w in SUSPICIOUS_WORDS),\n            "brand_in_subdomain": int(any(b in sub for b in BRANDS) and domain not in BRANDS),\n            "brand_in_path": int(any(b in path.lower() for b in BRANDS)),\n            "is_shortener": int(host.removeprefix("www.") in SHORTENERS),\n            "path_depth": path.count("/"), "query_params": len(parse_qsl(query)),\n            "host_entropy": shannon_entropy(host), "path_entropy": shannon_entropy(path),\n            "double_slash_in_path": int("//" in path),\n            "percent_encoded_count": len(re.findall(r"%[0-9a-fA-F]{2}", url)),\n            "is_punycode": int("xn--" in host),\n            "tld_length": len(tld),\n            "suspicious_tld": int(tld in SUSPICIOUS_TLDS),\n            "longest_host_token": max((len(t) for t in tokens), default=0),\n            "host_vowel_ratio": (sum(c in VOWELS for c in host_letters) / len(host_letters)) if host_letters else 0.0,\n            "max_repeated_char_run": _longest_run(url),\n            "file_extension_risky": int(ext in RISKY_EXT),\n        }\n    except ValueError:\n        return {name: 0 for name in FEATURE_NAMES}\n\n\ndef features_frame(urls) -> pd.DataFrame:\n    """Vectorise many URLs into a DataFrame with a stable column order."""\n    return pd.DataFrame([extract_features(u) for u in urls], columns=FEATURE_NAMES)\n\n\ndef risk_signals(raw_url: str) -> list[str]:\n    """Human-readable red flags for one URL. Explanatory only, not the model\'s decision."""\n    f = extract_features(raw_url)\n    checks = [\n        (f["has_ip_host"], "Host is a raw IP address instead of a domain name"),\n        (f["has_at_symbol"], "Contains \'@\', which can hide the real destination"),\n        (f["is_shortener"], "Uses a link shortener, so the final target is hidden"),\n        (f["brand_in_subdomain"], "Brand name appears in the subdomain of an unrelated domain"),\n        (f["suspicious_tld"], "Uses a top-level domain often seen in abuse"),\n        (f["is_punycode"], "Contains punycode (possible look-alike characters)"),\n        (f["suspicious_word_count"] >= 2, f"Contains {f[\'suspicious_word_count\']} sensitive words (login, verify, ...)"),\n        (f["hyphen_in_host"] >= 2, "Many hyphens in the host name"),\n        (f["subdomain_count"] >= 3, "Unusually many subdomains"),\n        (f["has_port"], "Uses an explicit non-default port"),\n        (f["double_slash_in_path"], "Double slash inside the path"),\n        (f["file_extension_risky"], "Points directly at an executable or script-like file"),\n        (f["url_length"] > 100, "Very long URL"),\n        (not str(raw_url).strip().lower().startswith("https://"), "Not using HTTPS"),\n    ]\n    return [msg for cond, msg in checks if cond]\n', 'cyberguard/data.py': '"""Dataset loading, label cleaning and leakage-aware splitting."""\nfrom __future__ import annotations\n\nfrom urllib.parse import urlparse\n\nimport pandas as pd\nfrom sklearn.model_selection import GroupShuffleSplit, train_test_split\n\nfrom .features import normalize, split_host\n\nSEED = 42\nLEGIT_LABELS = {"0", "benign", "legitimate", "legit", "good", "safe"}\nURL_COLS = ["url", "urls", "link"]\nLABEL_COLS = ["label", "type", "class", "status", "result"]\nDATE_COLS = ["date", "timestamp", "time", "first_seen", "submission_time", "created", "created_at"]\n\nDEMO_LEGIT = [\n    "https://www.google.com", "https://www.wikipedia.org/wiki/Python_(programming_language)",\n    "https://github.com/python/cpython", "https://www.python.org/downloads/",\n    "https://stackoverflow.com/questions/tagged/python", "https://www.bbc.com/news",\n    "https://docs.microsoft.com/en-us/learn/", "https://www.nasa.gov/missions",\n    "https://www.youtube.com/watch?v=abc123", "https://www.amazon.in/gp/bestsellers",\n    "https://www.linkedin.com/in/example", "https://developer.mozilla.org/en-US/docs/Web/JavaScript",\n    "https://www.khanacademy.org/math", "https://www.kernel.org", "https://pypi.org/project/pandas/",\n    "https://www.mozilla.org/en-US/firefox/", "https://scikit-learn.org/stable/",\n    "https://www.coursera.org/learn/machine-learning", "https://news.ycombinator.com",\n    "https://www.reddit.com/r/learnpython", "https://www.ubuntu.com/download", "https://www.gnu.org/licenses/",\n    "https://www.cloudflare.com/learning/", "https://www.w3schools.com/python/",\n]\nDEMO_MALICIOUS = [\n    "http://secure-login.paypal-verify-account.example-xyz.com/signin?id=99283",\n    "http://192.168.45.12/bank/login/update.php", "http://bit.ly/3xFr9aQ",\n    "http://account-update.secure-banking-portal.info/verify/confirm", "http://free-bonus-wallet.top/claim?user=123&ref=88",\n    "http://login.microsoft-account-verify.example-co.net/password/reset", "http://www.amaz0n-secure-signin.biz/ap/update",\n    "http://203.0.113.77:8080/login.html", "http://verify-your-account-now.click/secure//confirm",\n    "http://paypa1-support-update.example-site.org/login", "http://secure.bank-update-center.info/account/verify/login",\n    "http://apple-id-locked.example-help.xyz/unlock?session=ab12cd34ef56", "http://netflix-billing-update.example-pay.top/signin",\n    "http://user@fake-login-page.example.biz/verify", "http://confirm-password-reset.free-gift-card.example.ru/a/b/c/d",\n    "http://my-secure-account.verify-login-update.example.cc/banking", "http://dhl-parcel-tracking.example-delivery.link/confirm?id=5543",\n    "http://login-update-secure-bank-account.example-123.xyz", "http://198.51.100.9/wallet/verify/login", "http://tinyurl.com/y8abcd12",\n    "http://webmail-quota-verify.example-mail.info/update/password", "http://secure-signin.bank-account-confirm.example-net.com/x",\n    "http://irs-refund-claim.example-gov.top/verify/account", "http://whatsapp-gold-free-download.example.click/install",\n]\n\n\ndef demo_dataset() -> pd.DataFrame:\n    """Tiny hand-written set so the code runs out of the box. Scores on it mean nothing."""\n    return pd.DataFrame({\n        "url": DEMO_LEGIT + DEMO_MALICIOUS,\n        "label": [0] * len(DEMO_LEGIT) + [1] * len(DEMO_MALICIOUS),\n    })\n\n\ndef load_dataset(path: str, max_rows: int | None = 100_000) -> pd.DataFrame:\n    """Load a CSV with a URL column and a label/type column. Returns columns url, label (0/1)."""\n    raw = pd.read_csv(path)\n    cols = {c.lower(): c for c in raw.columns}\n    url_col = next((cols[c] for c in URL_COLS if c in cols), None)\n    label_col = next((cols[c] for c in LABEL_COLS if c in cols), None)\n    if url_col is None or label_col is None:\n        raise ValueError(f"Need a url column and a label/type column. Found: {list(raw.columns)}")\n    date_col = next((cols[c] for c in DATE_COLS if c in cols), None)\n    keep = [url_col, label_col] + ([date_col] if date_col else [])\n    df = raw[keep].rename(columns={url_col: "url", label_col: "raw_label", **({date_col: "date"} if date_col else {})})\n    if date_col:\n        df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)\n    df = df.dropna(subset=["url", "raw_label"])\n    df = df.drop_duplicates(subset="url")\n    df["label"] = df["raw_label"].astype(str).str.strip().str.lower().map(lambda v: 0 if v in LEGIT_LABELS else 1)\n    if max_rows and len(df) > max_rows:\n        df = df.sample(max_rows, random_state=SEED)\n    return df[["url", "label"] + (["date"] if date_col else [])].reset_index(drop=True)\n\n\ndef registered_domain(url: str) -> str:\n    try:\n        host = (urlparse(normalize(url)).hostname or "").lower()\n    except ValueError:  # malformed URLs (e.g. broken IPv6 brackets) exist in real data\n        host = str(url).lower()\n    _, domain, suffix = split_host(host)\n    return f"{domain}.{suffix}" if suffix else host\n\n\ndef split_data(df: pd.DataFrame, test_size: float = 0.2, group_by_domain: bool = True, time_split: bool = False):\n    """Train/test split. Grouping by domain keeps one site from landing on both sides (leakage).\n\n    Falls back to a plain stratified split when grouping is impossible (e.g. tiny data).\n    Returns (train_df, test_df).\n    """\n    if time_split:\n        if "date" not in df or df["date"].notna().sum() < 0.9 * len(df):\n            raise ValueError("--time-split needs a date/timestamp column with valid dates in your CSV")\n        ordered = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)\n        cut = int(len(ordered) * (1 - test_size))\n        train, test = ordered.iloc[:cut], ordered.iloc[cut:]\n        if train["label"].nunique() < 2 or test["label"].nunique() < 2:\n            raise ValueError("Time split left only one class on one side; use a larger or more mixed dataset")\n        return train.reset_index(drop=True), test.reset_index(drop=True)\n    if group_by_domain and len(df) >= 50:\n        groups = df["url"].map(registered_domain)\n        if groups.nunique() > 10:\n            gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)\n            tr, te = next(gss.split(df, df["label"], groups))\n            if df["label"].iloc[tr].nunique() == 2 and df["label"].iloc[te].nunique() == 2:\n                return df.iloc[tr].reset_index(drop=True), df.iloc[te].reset_index(drop=True)\n    train, test = train_test_split(df, test_size=test_size, stratify=df["label"], random_state=SEED)\n    return train.reset_index(drop=True), test.reset_index(drop=True)\n', 'cyberguard/ngram.py': '"""Hybrid model: character n-gram TF-IDF on the raw URL + scaled lexical features."""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy.sparse import csr_matrix, hstack\nfrom sklearn.base import BaseEstimator, ClassifierMixin\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\n\nfrom .features import FEATURE_NAMES\n\n\nclass NgramHybrid(ClassifierMixin, BaseEstimator):\n    """Expects a DataFrame with the lexical feature columns plus a raw `url` text column."""\n\n    def __init__(self, C: float = 1.0, ngram_range=(3, 5), max_features: int = 50_000):\n        self.C = C\n        self.ngram_range = ngram_range\n        self.max_features = max_features\n\n    def _matrix(self, X, fit: bool = False):\n        text, num = X["url"].astype(str), X[FEATURE_NAMES].to_numpy(dtype=float)\n        if fit:\n            self.tfidf_ = TfidfVectorizer(analyzer="char", ngram_range=self.ngram_range,\n                                          max_features=self.max_features, sublinear_tf=True)\n            self.scaler_ = StandardScaler()\n            return hstack([self.tfidf_.fit_transform(text), csr_matrix(self.scaler_.fit_transform(num))]).tocsr()\n        return hstack([self.tfidf_.transform(text), csr_matrix(self.scaler_.transform(num))]).tocsr()\n\n    def fit(self, X, y):\n        self.clf_ = LogisticRegression(C=self.C, max_iter=3000, class_weight="balanced")\n        self.clf_.fit(self._matrix(X, fit=True), y)\n        self.classes_ = self.clf_.classes_\n        return self\n\n    def predict_proba(self, X):\n        return self.clf_.predict_proba(self._matrix(X))\n\n    def predict(self, X):\n        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]\n', 'cyberguard/modeling.py': '"""Training, evaluation, threshold tuning, persistence and prediction."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,\n                             precision_recall_curve, precision_score, recall_score, roc_auc_score)\nfrom sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom .data import SEED, registered_domain\nfrom .features import FEATURE_NAMES, features_frame, risk_signals, scheme_free\nfrom .ngram import NgramHybrid\n\nTEXT_MODELS = {"N-gram Hybrid"}  # models that also read the raw URL text\n\n\ndef model_input(urls, text: bool = False) -> pd.DataFrame:\n    """Feature table; adds the raw `url` column for models that use character n-grams."""\n    urls = list(urls)\n    X = features_frame(urls)\n    if text:\n        X["url"] = [scheme_free(u) for u in urls]\n    return X\n\n\ndef build_models(seed: int = SEED) -> dict:\n    models = {\n        "Logistic Regression": Pipeline([\n            ("scaler", StandardScaler()),\n            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),\n        ]),\n        "Random Forest": RandomForestClassifier(\n            n_estimators=300, class_weight="balanced", min_samples_leaf=2, random_state=seed, n_jobs=-1),\n        "Gradient Boosting": HistGradientBoostingClassifier(\n            class_weight="balanced", random_state=seed),\n    }\n    models["N-gram Hybrid"] = NgramHybrid()\n    try:  # optional dependency\n        from xgboost import XGBClassifier\n        models["XGBoost"] = XGBClassifier(eval_metric="logloss", random_state=seed, n_jobs=-1)\n    except ImportError:\n        pass\n    return models\n\n\ndef metrics_at(y_true, y_prob, threshold: float = 0.5) -> dict:\n    y_pred = (np.asarray(y_prob) >= threshold).astype(int)\n    out = {\n        "Accuracy": accuracy_score(y_true, y_pred),\n        "Precision": precision_score(y_true, y_pred, zero_division=0),\n        "Recall": recall_score(y_true, y_pred, zero_division=0),\n        "F1": f1_score(y_true, y_pred, zero_division=0),\n    }\n    if len(set(y_true)) == 2:\n        out["ROC-AUC"] = roc_auc_score(y_true, y_prob)\n        out["PR-AUC"] = average_precision_score(y_true, y_prob)\n    return out\n\n\ndef pick_threshold(y_true, y_prob, min_recall: float = 0.95) -> float:\n    """Highest threshold that still keeps recall >= min_recall (fewest false alarms for that recall)."""\n    prec, rec, thr = precision_recall_curve(y_true, y_prob)\n    ok = [(p, t) for p, r, t in zip(prec[:-1], rec[:-1], thr) if r >= min_recall]\n    return float(max(ok, key=lambda x: (x[0], x[1]))[1]) if ok else 0.5\n\n\ndef train_and_compare(train_df: pd.DataFrame, test_df: pd.DataFrame, calibrate: bool = True):\n    """Fit every model, score on the held-out test set, return (results_df, fitted, probs, y_test)."""\n    full_tr, y_tr = model_input(train_df["url"], True), train_df["label"].to_numpy()\n    full_te, y_te = model_input(test_df["url"], True), test_df["label"].to_numpy()\n    n_splits = 5 if len(y_tr) >= 100 else 3\n    # Group CV by domain, like the train/test split, so CV scores are not inflated by same-site leakage.\n    groups = train_df["url"].map(registered_domain).to_numpy()\n    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)\n\n    rows, fitted, probs = [], {}, {}\n    for name, model in build_models().items():\n        X_tr, X_te = (full_tr, full_te) if name in TEXT_MODELS else (full_tr[FEATURE_NAMES], full_te[FEATURE_NAMES])\n        cv_f1 = cross_val_score(model, X_tr, y_tr, cv=cv, groups=groups, scoring="f1")\n        model.fit(X_tr, y_tr)\n        if calibrate and len(y_tr) >= 200:  # probabilities you can trust more\n            model = CalibratedClassifierCV(model, cv=3, method="isotonic").fit(X_tr, y_tr)\n        prob = model.predict_proba(X_te)[:, 1]\n        fitted[name], probs[name] = model, prob\n        rows.append({"Model": name, **metrics_at(y_te, prob),\n                     "CV F1 mean": cv_f1.mean(), "CV F1 std": cv_f1.std()})\n    results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)\n    return results, fitted, probs, y_te\n\n\ndef save_bundle(model, name: str, threshold: float, metrics: dict, path: str | Path, demo: bool = False,\n                baseline: dict | None = None, text_input: bool = False) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    joblib.dump({"model": model, "name": name, "features": FEATURE_NAMES,\n                 "threshold": threshold, "metrics": metrics, "demo": demo,\n                 "baseline": baseline or {}, "text_input": text_input}, path)\n    return path\n\n\ndef load_bundle(path: str | Path) -> dict:\n    bundle = joblib.load(path)\n    if bundle.get("features") != FEATURE_NAMES:\n        raise ValueError("Model was trained with a different feature set. Re-train it.")\n    return bundle\n\n\ndef predict_urls(bundle: dict, urls) -> pd.DataFrame:\n    """Score URLs. Returns url, probability, verdict, risk level."""\n    urls = list(urls)\n    prob = bundle["model"].predict_proba(model_input(urls, bundle.get("text_input", False)))[:, 1]\n    thr = bundle["threshold"]\n    level = np.select([prob >= max(thr, 0.8), prob >= thr, prob >= thr * 0.5], ["HIGH", "MEDIUM", "LOW"], "MINIMAL")\n    return pd.DataFrame({\n        "url": urls, "malicious_probability": prob.round(4),\n        "verdict": np.where(prob >= thr, "malicious", "legitimate"), "risk_level": level,\n    })\n\n\ndef local_explanation(bundle: dict, url: str, top: int = 5) -> list[dict]:\n    """Model-agnostic \'why this score?\': replace each feature with a typical (median) value\n    and measure how much the malicious probability drops. Positive = pushes towards malicious.\n    Cheap alternative to SHAP with no extra dependency."""\n    x = model_input([url], bundle.get("text_input", False))\n    base = bundle.get("baseline") or {}\n    p0 = float(bundle["model"].predict_proba(x)[0, 1])\n    variants = []\n    for f in FEATURE_NAMES:\n        v = x.copy()\n        v[f] = base.get(f, 0.0)\n        variants.append(v)\n    probs = bundle["model"].predict_proba(pd.concat(variants, ignore_index=True))[:, 1]\n    rows = [{"feature": f, "value": float(x[f].iloc[0]), "typical": float(base.get(f, 0.0)),\n             "impact": round(p0 - float(p), 4)} for f, p in zip(FEATURE_NAMES, probs)]\n    rows = [r for r in rows if r["impact"] != 0]\n    return sorted(rows, key=lambda r: abs(r["impact"]), reverse=True)[:top]\n\n\ndef explain(bundle: dict, url: str) -> dict:\n    row = predict_urls(bundle, [url]).iloc[0].to_dict()\n    row["signals"] = risk_signals(url)\n    row["top_features"] = local_explanation(bundle, url)\n    return row\n\n\ndef dump_report(results: pd.DataFrame, extra: dict, path: str | Path) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps({"results": results.round(4).to_dict("records"), **extra}, indent=2))\n\n\ndef error_analysis(test_df: pd.DataFrame, y_prob, threshold: float, top: int = 10) -> pd.DataFrame:\n    """Most confident mistakes: false positives (legit flagged) and false negatives (threats missed)."""\n    df = test_df[["url", "label"]].copy()\n    df["malicious_probability"] = np.asarray(y_prob).round(4)\n    df["predicted"] = (df["malicious_probability"] >= threshold).astype(int)\n    wrong = df[df["predicted"] != df["label"]].copy()\n    wrong["error_type"] = np.where(wrong["label"] == 0, "false_positive", "false_negative")\n    wrong["confidence"] = (wrong["malicious_probability"] - threshold).abs()\n    wrong = wrong.sort_values("confidence", ascending=False).groupby("error_type").head(top)\n    wrong["red_flags"] = wrong["url"].map(lambda u: "; ".join(risk_signals(u)))\n    return wrong.drop(columns=["confidence", "predicted"]).reset_index(drop=True)\n', 'cyberguard/plots.py': '"""Matplotlib helpers that save report images."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport seaborn as sns\nfrom sklearn.metrics import confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve\n\nfrom .features import FEATURE_NAMES\n\n\ndef _save(fig, out_dir, name):\n    if out_dir:\n        Path(out_dir).mkdir(parents=True, exist_ok=True)\n        fig.savefig(Path(out_dir) / name, dpi=150, bbox_inches="tight")\n\n\ndef confusion(y_true, y_prob, threshold, title, out_dir=None):\n    cm = confusion_matrix(y_true, (np.asarray(y_prob) >= threshold).astype(int), labels=[0, 1])\n    fig, ax = plt.subplots(figsize=(5, 4))\n    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["Legit", "Malicious"],\n                yticklabels=["Legit", "Malicious"])\n    ax.set(title=f"{title} (threshold {threshold:.2f})", xlabel="Predicted", ylabel="Actual")\n    _save(fig, out_dir, "confusion_matrix.png")\n    return fig\n\n\ndef roc_and_pr(y_true, probs: dict, out_dir=None):\n    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.5))\n    for name, p in probs.items():\n        fpr, tpr, _ = roc_curve(y_true, p)\n        a.plot(fpr, tpr, lw=2, label=f"{name} (AUC {roc_auc_score(y_true, p):.3f})")\n        pr, rc, _ = precision_recall_curve(y_true, p)\n        b.plot(rc, pr, lw=2, label=name)\n    a.plot([0, 1], [0, 1], "--", color="gray")\n    a.set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate"); a.legend(loc="lower right")\n    b.set(title="Precision-recall curve", xlabel="Recall", ylabel="Precision"); b.legend(loc="lower left")\n    fig.tight_layout()\n    _save(fig, out_dir, "roc_pr_curves.png")\n    return fig\n\n\ndef feature_importance(model, out_dir=None, top: int = 20):\n    est = getattr(model, "calibrated_classifiers_", [None])[0]\n    est = getattr(est, "estimator", model)\n    if not hasattr(est, "feature_importances_"):\n        return None\n    s = (__import__("pandas").Series(est.feature_importances_, index=FEATURE_NAMES)\n         .sort_values().tail(top))\n    fig, ax = plt.subplots(figsize=(7, 6))\n    s.plot(kind="barh", color="#e07a5f", ax=ax)\n    ax.set(title="Top feature importances", xlabel="Importance")\n    fig.tight_layout()\n    _save(fig, out_dir, "feature_importance.png")\n    return fig\n'}
os.makedirs("cyberguard", exist_ok=True)
for path, text in FILES.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
print("Package written:", ", ".join(sorted(FILES)))

In [ ]:
import sys, warnings
sys.path.insert(0, ".")
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from cyberguard import FEATURE_NAMES, extract_features, features_frame
from cyberguard.data import demo_dataset, load_dataset, split_data
from cyberguard.features import risk_signals
from cyberguard.modeling import (train_and_compare, pick_threshold, error_analysis, TEXT_MODELS, save_bundle,
                                 load_bundle, predict_urls, explain)
from cyberguard import plots
print(len(FEATURE_NAMES), "features ready")

## 2. Feature extraction
Each URL becomes numbers. Only the **text** of the URL is used, never the website.

In [ ]:
sample = "http://secure-login.paypal-verify.example-xyz.com/signin?id=99283"
display(pd.Series(extract_features(sample)).to_frame("value"))
print("\nRed flags:"); [print(" -", s) for s in risk_signals(sample)];

## 3. Load data
This looks for **`malicious_phish.csv`** in the Colab files panel (or the current folder). If it isn't there, a file picker opens: choose your CSV (columns `url` and `type`/`label`). Cancel the picker to use the tiny demo set (its scores are **not meaningful**).

In [ ]:
import os
MAX_ROWS = 50_000   # raise to 100_000+ if your Colab runtime has enough RAM and time
CSV_NAME = "malicious_phish.csv"
candidates = [CSV_NAME, f"/content/{CSV_NAME}", f"../{CSV_NAME}", f"data/{CSV_NAME}",
              f"/content/drive/MyDrive/{CSV_NAME}"]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    try:
        from google.colab import files
        print(f"Upload {CSV_NAME} (or any CSV with url + label/type columns). Cancel to use the demo set.")
        uploaded = files.upload()
        if uploaded:
            DATA_PATH = next(iter(uploaded))
    except ImportError:
        pass   # not running in Colab

DEMO = DATA_PATH is None
data = demo_dataset() if DEMO else load_dataset(DATA_PATH, max_rows=MAX_ROWS)
print("⚠️ DEMO MODE: scores are NOT meaningful" if DEMO else f"Using {DATA_PATH}")
print(f"{len(data):,} URLs")
print(data["label"].value_counts().rename({0: "legitimate", 1: "malicious"}))
display(data.head())

## 4. Split
Grouped by registered domain, so the same site never appears in both train and test (stops over-optimistic scores).

In [ ]:
TIME_SPLIT = False   # True: train on older URLs, test on newest (needs a date column in your CSV)
train_df, test_df = split_data(data, time_split=TIME_SPLIT)
print(len(train_df), "train /", len(test_df), "test")

## 5. Train and compare
Logistic Regression, Random Forest, Gradient Boosting (and XGBoost if installed), with cross-validation and probability calibration on larger data.

In [ ]:
results, fitted, probs, y_test = train_and_compare(train_df, test_df)
best = results.loc[0, "Model"]
display(results.round(4))
print("🏆 Best model by F1:", best)

## 6. Choose a decision threshold
In security, missing a threat is usually worse than a false alarm. Pick the threshold that keeps **recall ≥ 85%** while maximising precision. (On the Kaggle data, 95% recall meant about 31% false alarms; 85% gives about 11%.)

In [ ]:
MIN_RECALL = 0.85   # 0.95 catches more threats but raises false alarms a lot on real data
threshold = pick_threshold(y_test, probs[best], MIN_RECALL)
print(f"Threshold for recall >= {MIN_RECALL:.0%}: {threshold:.3f}")

## 7. Visualisations

In [ ]:
plots.confusion(y_test, probs[best], threshold, best, out_dir="reports")
plots.roc_and_pr(y_test, probs, out_dir="reports")
plots.feature_importance(fitted[best], out_dir="reports")
plt.show()

## 7b. Error analysis
The most confident mistakes. Study them: they show where the features fall short.

In [ ]:
os.makedirs("reports", exist_ok=True)
errors = error_analysis(test_df, probs[best], threshold)
errors.to_csv("reports/error_analysis.csv", index=False)
display(errors)

## 8. Check any URL
Shows the model verdict **and** plain-English red flags.

In [ ]:
bundle = {"model": fitted[best], "threshold": threshold, "text_input": best in TEXT_MODELS,
          "baseline": features_frame(train_df["url"]).median().to_dict()}
for url in ["http://secure-login.paypal-update.example-bank.com/verify?id=99283",
            "https://www.python.org/downloads/",
            "https://accounts.google.com/login"]:   # real/popular sites: this dataset has few famous benign domains, so false alarms are likely
    r = explain(bundle, url)
    print(f"{r['verdict'].upper():<11} {r['malicious_probability']:.1%} [{r['risk_level']}]  {url}")
    for s in r["signals"]: print("    -", s)

## 9. Save the model
Saved with its feature list and threshold; the CLI and Streamlit app load this file.

In [ ]:
path = save_bundle(fitted[best], best, threshold, results.loc[0].drop("Model").round(4).to_dict(),
                   "models/cyberguard_model.joblib", demo=DEMO,
                   baseline=bundle["baseline"], text_input=best in TEXT_MODELS)
print("Saved", path)

In [ ]:
# Optional (Colab): bundle the model and report images into one zip and download it
import shutil
shutil.make_archive("cyberguard_results", "zip", ".", "reports")
try:
    from google.colab import files
    files.download("cyberguard_results.zip")
    files.download("models/cyberguard_model.joblib")
except ImportError:
    print("Saved cyberguard_results.zip and models/cyberguard_model.joblib")

## 10. Limitations
- URL text only: no page content, domain age, SSL details or reputation lists.
- Words like `login` also appear on real sites, so false alarms happen.
- Results depend on the dataset; mixed sources can inflate scores. Report dataset name and size.
- Attackers change URLs, so models go stale. Re-train regularly.

## 11. Next steps
WHOIS/domain-age features · character n-gram TF-IDF · SHAP explanations · time-based validation · model monitoring